# 红利低波数据下载

本 notebook 只负责数据更新。参数来自 `config.yaml`；每次只启动一个下载任务。

In [ ]:
import os
import sys
from pathlib import Path

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists())
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import pandas as pd

from pyquant import load_config, load_dataset, update_dataset
from strategies.cross_sectional.dividend_low_vol.components import select_dividend_low_vol_download_symbols

strategy_dir = project_root / 'strategies/cross_sectional/dividend_low_vol'
config = load_config(strategy_dir / 'config.yaml')
start_date = pd.Timestamp(config['data']['start_date'])
end_date = pd.Timestamp(config['data']['end_date'])
pool = config['data']['pool']
lookback_date = start_date - pd.DateOffset(years=3)
dividend_start = lookback_date - pd.DateOffset(years=1) if (start_date.month == 12 and start_date.day <= 20) else lookback_date

## 日行情

先运行此单元格并等待完成，再生成分红和股本下载股票池。

In [ ]:
download_job = update_dataset(
    'stock_daily',
    start=lookback_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d'),
    pool=pool,
)

### 下载控制

In [ ]:
download_job.pause()
download_job.state

In [ ]:
download_job.resume()
download_job.state

In [ ]:
download_job.stop()
download_job.wait()
download_job.state

## 分红与股本下载股票池

按完整下载区间内至少 720 条有效行情筛选；结果覆盖 `pool`，供分红和股本共用。

In [ ]:
price = load_dataset(
    'stock_daily',
    start=lookback_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d'),
)
pool = select_dividend_low_vol_download_symbols(price, end_date, config)
if not pool:
    raise ValueError('No symbols have at least 720 valid prices in the download range')
print(f'Dividend/share download pool: {len(pool)} symbols')

## 分红数据

In [ ]:
download_job = update_dataset(
    'dividend',
    start=dividend_start.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d'),
    pool=pool,
)

## 季度总股本

In [ ]:
download_job = update_dataset(
    'stock_profit_quarterly',
    start=lookback_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d'),
    pool=pool,
)